In [2]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

## 1. Règle d'affectation MR-Sort

$p$ catégories sont séparées par $p-1$ profils. Pour $p = 3$ :
- $b^1$ est la frontière entre $C^1$ et $C^2$, et $b^2$ la frontière entre $C^2$ et $C^3$, avec $b^1 \le b^2$ critère par critère ;
- $x \in C^1$ si $x$ ne surclasse pas $b^1$ ;
- $x \in C^2$ si $x$ surclasse $b^1$ mais pas $b^2$ ;
- $x \in C^3$ si $x$ surclasse $b^2$.

Les profils étant emboîtés, un objet qui surclasse $b^2$ surclasse aussi $b^1$. La classe de $x$ est donc le nombre de profils qu'il surclasse. Dans le code, les classes vont de $0$ à $p-1$ et `b[h]` sépare la classe `h` de la classe `h+1`.

In [4]:
class MRSort:
    def __init__(self, w, lam, b):
        self.w = np.asarray(w, dtype=float)
        self.lam = float(lam)
        self.b = np.atleast_2d(np.asarray(b, dtype=float))   # (p-1, n)

    def predict(self, X):
        coalitions = (np.atleast_2d(X)[:, None, :] >= self.b[None, :, :]) @ self.w   # (m, p-1)
        return (coalitions >= self.lam - 1e-9).sum(axis=1)

    def __repr__(self):
        with np.printoptions(precision=3, suppress=True):
            return f"w   = {self.w}\nlam = {self.lam:.3f}\nb   =\n{self.b}"

## 2. Programme mathématique (questions v à viii)

On prend $M = x_{\max} - x_{\min} + 1$, et $\varepsilon > 0$ petit pour modéliser les inégalités strictes.

Variables :
- $w_i$, $\lambda$, $b^h_i$ : les paramètres du modèle ;
- $\delta^h_i(x) \in \{0,1\}$ : vaut 1 si et seulement si $x_i \ge b^h_i$ ;
- $w^h_i(x) = w_i\,\delta^h_i(x)$ ;
- $\gamma(x) \in \{0,1\}$ : vaut 1 si $x$ est bien restitué.

$$
\begin{aligned}
\max \ & \textstyle\sum_{x \in \mathcal{L}} \gamma(x) \\
\text{s.c.}\ & \textstyle\sum_i w_i = 1, \quad \lambda \in [0, 1], \quad b^h_i \ge b^{h-1}_i \\
& M(\delta^h_i(x) - 1) \le x_i - b^h_i \le M\,\delta^h_i(x) - \varepsilon && \text{(q. v)} \\
& 0 \le w^h_i(x) \le w_i, \quad \delta^h_i(x) + w_i - 1 \le w^h_i(x) \le \delta^h_i(x) && \text{(q. vi)} \\
& \textstyle\sum_i w^{h-1}_i(x) \ge \lambda - 2\,(1 - \gamma(x)) && x \in C^h,\ h \ge 2 \\
& \textstyle\sum_i w^{h}_i(x) + \varepsilon \le \lambda + 2\,(1 - \gamma(x)) && x \in C^h,\ h \le p-1
\end{aligned}
$$

L'objectif traduit « restituer au mieux » : si $\mathcal{L}$ n'est pas représentable exactement par un MR-Sort, on maximise le nombre d'objets bien affectés. Pour un objet de $C^h$, seuls les profils $b^{h-1}$ et $b^h$ interviennent.

GUROBI satisfait les contraintes à une tolérance près (environ $10^{-6}$). Après résolution, on recentre donc $b$ et $\lambda$ de $-\varepsilon/2$, c'est-à-dire au milieu de l'écart laissé par les inégalités strictes.

In [5]:
def learn_mrsort(X, y, p, x_min, x_max, time_limit=120):
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=int)
    m, n = X.shape
    M, eps = x_max - x_min + 1, 1e-3

    mdl = gp.Model("inv-MR-Sort")
    mdl.Params.OutputFlag = 0
    mdl.Params.TimeLimit = time_limit
    mdl.Params.IntFeasTol = 1e-9

    w = mdl.addVars(n, lb=0, ub=1, name="w")
    lam = mdl.addVar(lb=0, ub=1, name="lambda")
    b = mdl.addVars(p - 1, n, lb=x_min, ub=x_max + eps, name="b")
    gamma = mdl.addVars(m, vtype=GRB.BINARY, name="gamma")

    mdl.addConstr(w.sum() == 1)
    mdl.addConstrs(b[h, i] >= b[h - 1, i] for h in range(1, p - 1) for i in range(n))

    def coalition(k, h):
        # sum_i w_i^h(x_k), avec delta_i^h(x_k) = 1 <=> x_ki >= b^h_i
        d = mdl.addVars(n, vtype=GRB.BINARY)
        wx = mdl.addVars(n, lb=0, ub=1)
        for i in range(n):
            mdl.addConstr(X[k, i] - b[h, i] >= M * (d[i] - 1))
            mdl.addConstr(X[k, i] - b[h, i] + eps <= M * d[i])
            mdl.addConstr(wx[i] <= w[i])
            mdl.addConstr(wx[i] <= d[i])
            mdl.addConstr(wx[i] >= d[i] + w[i] - 1)
        return wx.sum()

    for k in range(m):
        if y[k] >= 1:          # surclasse le profil inférieur de sa classe
            mdl.addConstr(coalition(k, y[k] - 1) >= lam - 2 * (1 - gamma[k]))
        if y[k] <= p - 2:      # ne surclasse pas le profil supérieur de sa classe
            mdl.addConstr(coalition(k, y[k]) + eps <= lam + 2 * (1 - gamma[k]))

    mdl.setObjective(gamma.sum(), GRB.MAXIMIZE)
    mdl.optimize()

    model = MRSort([w[i].X for i in range(n)], lam.X - eps / 2,
                   [[b[h, i].X - eps / 2 for i in range(n)] for h in range(p - 1)])
    return model, mdl

## 3. Validation sur Dataset0

Le fichier est séparé par des tabulations : une colonne d'identifiant, les $n$ critères, puis la catégorie. On suppose que la catégorie 1 est la moins bonne. Les évaluations sont dans $[0, 100]$.

In [7]:
def load_dataset(path):
    rows = [line.split("\t") for line in open(path).read().splitlines()[1:] if line.strip()]
    data = np.array([[float(v) for v in r[1:] if v.strip()] for r in rows])
    X, labels = data[:, :-1], data[:, -1].astype(int)
    classes = np.unique(labels)
    return X, np.searchsorted(classes, labels), X.shape[1], len(classes)

X0, y0, n0, p0 = load_dataset("dataset0.txt")
print(f"|L| = {len(X0)}, n = {n0}, p = {p0}")

model0, mdl0 = learn_mrsort(X0, y0, p0, x_min=0, x_max=100)
print(model0)
print(f"Objets restitués : {int(mdl0.ObjVal)}/{len(X0)} (statut GUROBI {mdl0.Status}, {mdl0.Runtime:.2f} s)")
print(f"Contrôle avec la règle MR-Sort : {np.mean(model0.predict(X0) == y0):.3f}")


X = [[11, 12, 13, 12],
    [13, 12, 15, 10],
    [11, 13, 11, 13],
    [12, 12, 16, 12]]
y = [1, 0, 0, 1]
n = 4
p = 2
model , mdl = learn_mrsort(X, y, p, x_min=0, x_max=20)
print(model)
print(f"Objets restitués : {int(mdl.ObjVal)}/{len(X)} (statut GUROBI {mdl.Status}, {mdl.Runtime:.2f} s)")
print(f"Contrôle avec la règle MR-Sort : {np.mean(model.predict(X) == y):.3f}")

|L| = 60, n = 5, p = 3
Set parameter Username
Set parameter LicenseID to value 2870711
Academic license - for non-commercial use only - expires 2027-10-02
w   = [0.331 0.333 0.001 0.002 0.332]
lam = 0.666
b   =
[[36.999 34.    56.999 34.999 35.999]
 [53.999 52.999 56.999 56.    48.   ]]
Objets restitués : 60/60 (statut GUROBI 2, 0.81 s)
Contrôle avec la règle MR-Sort : 1.000
w   = [0.998 0.    0.001 0.001]
lam = 0.001
b   =
[[13.    11.999 11.    10.   ]]
Objets restitués : 4/4 (statut GUROBI 2, 0.00 s)
Contrôle avec la règle MR-Sort : 1.000


## 4. Test de performance

À chaque essai :
1. on génère un modèle ground truth $\mathcal{M}'$ : poids tirés uniformément sur le simplexe, $\lambda \sim \mathcal{U}[0, 1]$, et profils obtenus en triant des tirages uniformes (ils sont donc emboîtés) ;
2. on classe avec $\mathcal{M}'$ un learning set $\mathcal{L}$ d'objets uniformes, ce qui donne $\mathcal{L}$ étiqueté ;
3. on apprend $\mathcal{M}^*$ ;
4. sur un test set $\mathcal{T}$, la qualité est la proportion d'objets affectés à la même classe par $\mathcal{M}'$ et $\mathcal{M}^*$.

On répète 20 fois avec la même configuration que Dataset0 : $n = 5$, $p = 3$, évaluations dans $[0, 100]$. On rejette les ground truth qui laissent une catégorie vide dans $\mathcal{L}$.

In [5]:
def random_mrsort(n, p, x_min, x_max, rng):
    return MRSort(rng.dirichlet(np.ones(n)), rng.uniform(0, 1),
                  np.sort(rng.uniform(x_min, x_max, size=(p - 1, n)), axis=0))

def run_test(n, p, n_train, n_test, x_min, x_max, rng):
    while True:
        gt = random_mrsort(n, p, x_min, x_max, rng)
        X_train = rng.uniform(x_min, x_max, size=(n_train, n))
        y_train = gt.predict(X_train)
        if len(np.unique(y_train)) == p:
            break
    learned, mdl = learn_mrsort(X_train, y_train, p, x_min, x_max)
    X_test = rng.uniform(x_min, x_max, size=(n_test, n))
    return np.mean(gt.predict(X_test) == learned.predict(X_test)), mdl.Runtime

rng = np.random.default_rng(0)
N_RUNS, N, P, N_TRAIN, N_TEST = 20, 5, 3, 100, 10_000

results = np.array([run_test(N, P, N_TRAIN, N_TEST, 0, 100, rng) for _ in range(N_RUNS)])
quality, runtime = results[:, 0], results[:, 1]
print("Qualité par essai :", np.round(quality, 3))
print(f"Qualité moyenne : {quality.mean():.4f} ± {quality.std():.4f} (min {quality.min():.4f})")
print(f"Temps moyen de résolution : {runtime.mean():.2f} s (max {runtime.max():.2f} s)")

Qualité par essai : [0.966 0.913 0.946 0.956 0.984 0.911 0.934 0.981 0.935 0.951 0.939 0.955
 0.952 0.892 0.951 0.954 0.878 0.896 0.912 0.925]
Qualité moyenne : 0.9365 ± 0.0282 (min 0.8783)
Temps moyen de résolution : 0.62 s (max 1.37 s)
